In [3]:
import os                                   #imports
import pandas as pd
import numpy as np
import transformers
from transformers import AutoTokenizer
from sklearn.model_selection import train_test_split
from datasets import Dataset,DatasetDict

In [4]:
df = pd.read_csv("/content/train.csv")

In [5]:
df.drop('id',axis=1,inplace = True)

In [6]:
raw_ds = Dataset.from_pandas(df)

In [7]:
raw_ds = raw_ds.train_test_split(test_size = 0.2)
raw_ds

DatasetDict({
    train: Dataset({
        features: ['comment_text', 'toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate'],
        num_rows: 127656
    })
    test: Dataset({
        features: ['comment_text', 'toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate'],
        num_rows: 31915
    })
})

In [8]:
train_val = raw_ds['train'].train_test_split(test_size = 0.1)


In [9]:
final_ds = DatasetDict({
    "train":train_val['train'],
    "validation":train_val['test'],
    "test":raw_ds['test']
  })

In [10]:
final_ds

DatasetDict({
    train: Dataset({
        features: ['comment_text', 'toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate'],
        num_rows: 114890
    })
    validation: Dataset({
        features: ['comment_text', 'toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate'],
        num_rows: 12766
    })
    test: Dataset({
        features: ['comment_text', 'toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate'],
        num_rows: 31915
    })
})

In [11]:

tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [12]:
label_cols = ["toxic","severe_toxic","obscene","threat","insult","identity_hate"]

In [13]:
def create_labels(example):
    example["labels"] = [float(example[col]) for col in label_cols]
    return example
final_ds = final_ds.map(create_labels)

Map:   0%|          | 0/114890 [00:00<?, ? examples/s]

Map:   0%|          | 0/12766 [00:00<?, ? examples/s]

Map:   0%|          | 0/31915 [00:00<?, ? examples/s]

In [14]:
final_ds

DatasetDict({
    train: Dataset({
        features: ['comment_text', 'toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate', 'labels'],
        num_rows: 114890
    })
    validation: Dataset({
        features: ['comment_text', 'toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate', 'labels'],
        num_rows: 12766
    })
    test: Dataset({
        features: ['comment_text', 'toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate', 'labels'],
        num_rows: 31915
    })
})

In [15]:
def tokenize(example):
  return tokenizer(example['comment_text'],truncation = True)
ips = final_ds.map(tokenize,batched = True)

Map:   0%|          | 0/114890 [00:00<?, ? examples/s]

Map:   0%|          | 0/12766 [00:00<?, ? examples/s]

Map:   0%|          | 0/31915 [00:00<?, ? examples/s]

In [17]:
ips = ips.remove_columns(['comment_text'] + label_cols)

In [23]:
ips

DatasetDict({
    train: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 114890
    })
    validation: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 12766
    })
    test: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 31915
    })
})

In [20]:
from transformers import DataCollatorWithPadding

In [21]:
data_collator = DataCollatorWithPadding(tokenizer = tokenizer)

In [24]:
from transformers import AutoModelForSequenceClassification
model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=6,
    problem_type="multi_label_classification"
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",

    # training
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,

    # evaluation
    evaluation_strategy="epoch",
    save_strategy="epoch",

    # optimization
    optim="adamw_torch",
    learning_rate=2e-5,
    weight_decay=0.01,

    # logging
    logging_dir="./logs",
    logging_steps=100,

    # important
    load_best_model_at_end=True,
    metric_for_best_model="f1",
)

In [ ]:
from sklearn.metrics import f1_score
def find_best_thresholds(logits, labels):
    probs = 1 / (1 + np.exp(-logits))
    best_thresh = []
    for i in range(labels.shape[1]):
        best_f1 = 0
        best_t = 0.5

        for t in np.arange(0.1, 0.9, 0.05):
            preds = (probs[:, i] > t).astype(int)
            f1 = f1_score(labels[:, i], preds)

            if f1 > best_f1:
                best_f1 = f1
                best_t = t

        best_thresh.append(best_t)
    return best_thresh
def get_compute_metrics(eval_pred):
  def compute_metrics(eval_pred,best_thresh):
    logits,labels = eval_pred
    probs = 1/(1+np.exp(-logits))
    preds = (probs>best_thresh).astype(int)
    macro_f1 = f1_score(labels,preds,average = 'macro')
    return {'macro_f1':macro_f1}
  return compute_metrics